# Training pipeline

## 1. Intro markdown

### VJeaNETte Training Pipeline

This notebook reproduces the full model training pipeline:

1. Dataset generation
2. Tensor serialization
3. Train/validation/test split
4. UNet training
5. Validation metrics
6. Model checkpointing

The notebook demonstrates how VJeaNETte models were trained.

## 2. Imports

In [1]:
import os
import torch
import tomli
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

from vjeanette.preprocessing import (
    VDJDataset,
    dataset_to_pt
)

from vjeanette.model.model import UNet1D_Embed

## 3. Configs

## Configuration

Training parameters are loaded directly from `pyproject.toml`.

In [2]:
pt_path = '../data/train.pt'
test_split = 0.3
val_split = 0.1
batch_size = 256
model_name = '../weights/model_custom.pth'
lr = 0.0003
embed_dim = 32
NUM_EPOCHS = 130
early_stopping_patience = 30

## 4. Device setup

### CUDA / device initialization

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


## 5. Dataset loading

### Tensor dataset loading

In [4]:
data = torch.load(pt_path)

X = data["X"]
y = data["y"]

print(X.shape)
print(y.shape)

/scratch/ipykernel_4130469/2908245703.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pt_path)


torch.Size([474744, 512])
torch.Size([474744, 3, 512])


## 7. Train/val/test split

### Dataset split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_split,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=val_split,
    random_state=42
)

## 8. Dataloaders

### PyTorch dataloaders

In [6]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=batch_size
)

## 9. Loss functions

### Auxiliary losses and metrics

The training procedure combines:

- positional BCE loss
- existence loss
- recall metric

### Existence loss

In [7]:
def existence_loss(logits, target):

    pred = torch.sigmoid(logits).max(dim=1).values
    tgt = target.max(dim=1).values

    return nn.BCELoss()(pred, tgt)

### Recall metric

In [8]:
def recall_metric(logits, target, thr=0.01):

    probs = torch.sigmoid(logits)

    pred_exist = (probs.max(dim=1).values > thr)
    tgt_exist = (target.max(dim=1).values > 0)

    tp = (pred_exist & tgt_exist).sum().item()
    fn = (~pred_exist & tgt_exist).sum().item()

    if tp + fn == 0:
        return 0

    return tp / (tp + fn)

## 10. Model initialization

### Model initialization

In [9]:
model = UNet1D_Embed(embed_dim=embed_dim).to(device)

criterion_v = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([2.0]).to(device)
)

criterion_j = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([2.0]).to(device)
)

criterion_cdr = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([2.0]).to(device)
)

optimizer = optim.Adam(
    model.parameters(),
    lr=lr
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=15,
    factor=0.5
)

## 11. Training loop

### Training parameters

In [10]:
best_val_loss = float("inf")

early_stopping_counter = 0

### Epoch loop

In [11]:
for epoch in range(NUM_EPOCHS):
        model.train()
        total_v_loss, total_j_loss, total_cdr_loss = 0.0, 0.0, 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).long()
            batch_y_cdr = batch_y[:,0,:].to(device).float()
            batch_y_v = batch_y[:,1,:].to(device).float()
            batch_y_j = batch_y[:,2,:].to(device).float()
            optimizer.zero_grad()
            outputs_v, outputs_j, outputs_cdr = model(batch_x)
            loss_v = criterion_v(outputs_v, batch_y_v)
            loss_j = criterion_j(outputs_j, batch_y_j)
            loss_cdr = criterion_cdr(outputs_cdr, batch_y_cdr)

        # existence loss
            loss_v_exist = existence_loss(outputs_v, batch_y_v)
            loss_j_exist = existence_loss(outputs_j, batch_y_j)
            loss_cdr_exist = existence_loss(outputs_cdr, batch_y_cdr)
            total_loss = (
                loss_v + loss_j + loss_cdr
                + 2.0 * (loss_v_exist + loss_j_exist + loss_cdr_exist) 
            )
            total_loss.backward()
            optimizer.step()

            total_v_loss += loss_v.item()
            total_j_loss += loss_j.item()
            total_cdr_loss += loss_cdr.item()

        # --- Validation ---
        model.eval()
        epoch_val_loss_v, epoch_val_loss_j,  epoch_val_loss_cdr = 0.0, 0.0, 0.0
        recall_v_total = 0
        recall_j_total = 0
        recall_cdr_total = 0
        count = 0
        with torch.no_grad():
            for val_x, val_y in val_loader:
                val_x=val_x.to(device).long()
                val_y_v = val_y[:,1,:].to(device).float()
                val_y_j = val_y[:,2,:].to(device).float()
                val_y_cdr = val_y[:,0,:].to(device).float()
                val_outputs_v, val_outputs_j, val_outputs_cdr = model(val_x)
                loss_v = criterion_v(val_outputs_v, val_y_v)
                loss_j = criterion_j(val_outputs_j, val_y_j)
                loss_cdr = criterion_cdr(val_outputs_cdr, val_y_cdr)
                recall_v_total += recall_metric(val_outputs_v, val_y_v)
                recall_j_total += recall_metric(val_outputs_j, val_y_j)
                recall_cdr_total += recall_metric(val_outputs_cdr, val_y_cdr)
                count+=1
                epoch_val_loss_v += loss_v.item()
                epoch_val_loss_j += loss_j.item()
                epoch_val_loss_cdr += loss_cdr.item()

        train_loss_v = total_v_loss / len(train_loader)
        train_loss_j = total_j_loss / len(train_loader)
        train_loss_cdr = total_cdr_loss / len(train_loader)
        val_loss_v = epoch_val_loss_v / len(val_loader)
        val_loss_j = epoch_val_loss_j / len(val_loader)
        val_loss_cdr = epoch_val_loss_cdr / len(val_loader)

        total_val_loss = val_loss_v + val_loss_j + val_loss_cdr

        # Scheduler step
        scheduler.step(total_val_loss)

        print(f"[{epoch+1}/{NUM_EPOCHS}] "
              f"Train Loss - V: {train_loss_v:.4f}, J: {train_loss_j:.4f}, CDR: {train_loss_cdr:.4f}| "
              f"Val Loss - V: {val_loss_v:.4f}, J: {val_loss_j:.4f}, CDR: {val_loss_cdr:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        print("VAL RECALL V:", recall_v_total/count)
        print("VAL RECALL J:", recall_j_total/count)
        print("VAL RECALL CDR:", recall_cdr_total/count)


        # --- Early stopping ---
        if total_val_loss < best_val_loss:
            best_val_loss = total_val_loss
            best_model_weights = model.state_dict().copy()
            early_stopping_counter = 0
            print(f"*** Best model updated ***")
        else:
            early_stopping_counter += 1
            print(f"Early stopping counter: {early_stopping_counter}/{early_stopping_patience}")
            if early_stopping_counter >= early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break


[1/130] Train Loss - V: 0.3657, J: 0.4621, CDR: 0.5128| Val Loss - V: 0.3587, J: 0.2629, CDR: 0.2426 | LR: 0.000300
VAL RECALL V: 1.0
VAL RECALL J: 1.0
VAL RECALL CDR: 1.0
*** Best model updated ***
[2/130] Train Loss - V: 0.0930, J: 0.1242, CDR: 0.1262| Val Loss - V: 0.6101, J: 0.1981, CDR: 0.1545 | LR: 0.000300
VAL RECALL V: 0.24732971189982464
VAL RECALL J: 0.4370283308888199
VAL RECALL CDR: 0.9989528244073435
Early stopping counter: 1/30
[3/130] Train Loss - V: 0.0635, J: 0.0472, CDR: 0.0711| Val Loss - V: 0.6950, J: 0.2410, CDR: 0.1803 | LR: 0.000300
VAL RECALL V: 0.35891866447289356
VAL RECALL J: 0.36720354236799724
VAL RECALL CDR: 0.41358575871641334
Early stopping counter: 2/30
[4/130] Train Loss - V: 0.0567, J: 0.0453, CDR: 0.0542| Val Loss - V: 0.4230, J: 0.1433, CDR: 0.1119 | LR: 0.000300
VAL RECALL V: 0.8035550164128639
VAL RECALL J: 0.7869395908267556
VAL RECALL CDR: 0.8383418651636287
*** Best model updated ***
[5/130] Train Loss - V: 0.0528, J: 0.0452, CDR: 0.0449| Val L

## 12. Validation metrics

In [12]:
print("VAL RECALL V:", recall_v_total/count)
print("VAL RECALL J:", recall_j_total/count)
print("VAL RECALL CDR:", recall_cdr_total/count)

VAL RECALL V: 0.9926797115474209
VAL RECALL J: 0.9906370854372605
VAL RECALL CDR: 0.9925802182966869


## 13. Checkpoint saving

### Model checkpointing

In [13]:
model.load_state_dict(best_model_weights)

torch.save(
    model.state_dict(),
    model_name
)

print("Training completed.")

Training completed.


## 14. Final evoluation

In [14]:
print("Best validation loss:", best_val_loss)

Best validation loss: 0.04896239513316407


## 15. Visualization

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(train_losses)
plt.plot(val_losses)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(["train", "val"])